In [36]:
import pandas as pd
import numpy as np
import ast
import re
import time

from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader
from IPython.utils.sysinfo import encoding

from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [37]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [38]:
data = pd.read_csv('/content/data_no_keywords.csv')

In [40]:
print(f"Всего статей: {len(data)}")
print(f"Тематик: {data['category'].nunique()}")
print("Тематики и количество статей")
data['category'].value_counts()

Всего статей: 13238
Тематик: 10
Тематики и количество статей


,count
category,
Бэкенд,1796
AI и ML,1781
Научпоп,1651
Менеджмент,1541
Администрирование,1483
Информационная безопасность,1468
Фронтенд,1053
Маркетинг и контент,937
Геймдев,934


In [41]:
def data_prep(text, title, topics):
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+", " ", text)
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text).strip()

    if isinstance(topics, str):
        topics = ast.literal_eval(topics)

    cleaned_topics = [topic.replace(" ", "") for topic in topics]
    topics_str = " ".join(cleaned_topics)

    final_text = f'{title} [SEP] {topics_str} [SEP] {text}'
    return final_text

In [42]:
data['full_text'] = data.apply(lambda x: data_prep(x['full_text'], x['title'], x['topics']), axis=1)

In [43]:
train_df, test_df = train_test_split(data, test_size=0.1, random_state=42, stratify=data['category'])

In [44]:
class HabrDataset(Dataset):
  def __init__(self, text, labels, tokenizer, max_len=256):
    self.text = text
    self.labels = labels
    self.tokenizer = tokenizer
    self.max_len = max_len

  def __len__(self):
    return len(self.text)

  def __getitem__(self, idx):
    text = str(self.text[idx])
    label = self.labels[idx]

    encoding = self.tokenizer(
      text,
      padding='max_length',
      truncation=True,
      max_length=self.max_len,
      return_tensors='pt'
    )

    return {
      'input_ids': encoding['input_ids'].flatten(),
      'attention_mask': encoding['attention_mask'].flatten(),
      'labels': torch.tensor(label, dtype=torch.long)
    }

In [45]:
def creating_dataloader(tokenizer, batch_size = 16, max_len=256):
  train_dataset = HabrDataset(train_texts, train_labels, tokenizer, max_len)
  test_dataset = HabrDataset(test_texts, test_labels, tokenizer, max_len)

  train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2
  )

  val_loader = DataLoader(
      test_dataset,
      batch_size=batch_size,
      shuffle=False,
      num_workers=2
  )
  return train_loader, val_loader

In [46]:
label_names = sorted(train_df['category'].unique())
num_labels = len(label_names)

label_to_id = {label: i for i, label in enumerate(label_names)}
id_to_label = {i: label for label, i in label_to_id.items()}

train_texts = train_df['full_text'].tolist()
train_labels = [label_to_id[label] for label in train_df['category']]

test_texts = test_df['full_text'].tolist()
test_labels = [label_to_id[label] for label in test_df['category']]

In [47]:
def speed_test(model_path, model, tokenizer, val_loader, device):
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=False))
    model.to(device)
    model.eval()

    times = []
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            start = time.time()
            outputs = model(**batch)
            times.append((time.time() - start) * 1000)

            if len(times) >= 50:
                break

    return np.mean(times) / val_loader.batch_size


# DeepPavlov/rubert-base-cased

In [48]:
rubert_model_name = "DeepPavlov/rubert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(rubert_model_name)

model_bert = AutoModelForSequenceClassification.from_pretrained(
    rubert_model_name,
    num_labels=num_labels
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you

In [49]:
train_loader, val_loader = creating_dataloader(tokenizer, 16)

In [57]:
bert_time = speed_test('/content/best_model_BERT.pt', model_bert,
                       tokenizer, val_loader,device)
print(f"ruBERT: {bert_time:.6f} мс/текст\n")

ruBERT: 1.520755 мс/текст



# Tochka-AI/ruRoPEBert-classic-base-2k

In [ ]:
robert_model_name = "Tochka-AI/ruRoPEBert-classic-base-2k"
tokenizer = AutoTokenizer.from_pretrained(robert_model_name)

robert_model = AutoModelForSequenceClassification.from_pretrained(
    robert_model_name,
    num_labels=num_labels
)

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: Tochka-AI/ruRoPEBert-classic-base-2k
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 
bert.embeddings.position_embeddings.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Co

In [ ]:
train_loader, val_loader = creating_dataloader(tokenizer, 16, 384)

In [59]:
robert_time = speed_test('/content/best_model_robert.pt', robert_model,
                         tokenizer, val_loader,device)
print(f"roBERT: {robert_time:.2f} мс/текст\n")

roBERT: 1.51 мс/текст



# deepvk/RuModernBERT-base

In [60]:
rumod_bert_model_name = "deepvk/RuModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(rumod_bert_model_name)

rumod_bert_model = AutoModelForSequenceClassification.from_pretrained(
    rumod_bert_model_name,
    num_labels=num_labels
)

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: deepvk/RuModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [61]:
train_loader, val_loader = creating_dataloader(tokenizer, 16, 384)

In [64]:
modern_time = speed_test('/content/best_model_rurobert.pt',
                         rumod_bert_model, tokenizer, val_loader, device)
print(f"rumodernPEBert: {modern_time:.2f} мс/текст\n")

rumodernPEBert: 3.62 мс/текст



In [65]:
print("="*40)
print("ИТОГО:")
print(f"ruBERT:        {bert_time:.2f} мс/текст")
print(f"ruModernBERT:  {modern_time:.2f} мс/текст")
print(f"ruRoPEBert:    {robert_time:.2f} мс/текст")

ИТОГО:
ruBERT:        1.52 мс/текст
ruModernBERT:  3.62 мс/текст
ruRoPEBert:    1.51 мс/текст
